# FHIR on RAG

This notebook is loading FHIR resources into a vector store and then using that to help prompt an LLM to answer questions about the data. To do that, it first flattens the FHIR resources into text files. It then uses [LlamaIndex](https://www.llamaindex.ai/) to load the text files into an in-memory vector store. Then it calls out to a LLama 3 running locally using [Ollama](https://ollama.ai/) using different [strategies](https://docs.llamaindex.ai/en/stable/module_guides/querying/response_synthesizers/root.html) for combining the FHIR with the question into the prompt.

In [28]:
# Some constants to use throughout 

in_file_glob = './working/raw_fhir'
flat_file_path = './working/flat'
vector_store_file_path = './working/vector_store'

## Flatten FHIR

This is going to read in any JSON files in the `in_file_glob`. It assumes that each file is a FHIR Bundle. It will first pull out the Patient resource and extract some key information, like name, from it to include in the text files it will create per resource. This helps the RAG know which patient a resource goes with. It then flattens each resource in the bundle. 


This creates a semi-english version of the resource that can be turned into a vector by the embedding. 

**To use this project,** you will need to create the working and raw_fhir directories and populate raw_fhir with FHIR Bundles. I used [Synthea](https://synthea.mitre.org/) to generate synthetic data in my testing.

In [40]:
import json
import os
from typing import Dict, Any, List


def extract_patient_name(resource: Dict[str, Any]) -> str:
    """
    Extract full patient name from a FHIR Patient resource.
    """
    if resource.get("resourceType", "").lower() != "patient":
        return ""

    name_list = resource.get("name", [])
    if not name_list:
        return ""

    # Prefer the name with use == "official", or fallback to the first
    official_name = next((n for n in name_list if n.get("use") == "official"), name_list[0])

    given = official_name.get("given", [])
    family = official_name.get("family", "")
    prefix = official_name.get("prefix", [])

    full_name = " ".join(prefix + given + [family]).strip()
    return full_name


def fhir_resource_to_text(resource: Dict[str, Any], patient_name_map: Dict[str, str]) -> str:
    """
    Convert a FHIR resource to a plain English sentence.
    """
    resource_type = resource.get("resourceType", "").lower()
    subject_ref = resource.get("subject", {}).get("reference", "")
    patient_name = patient_name_map.get(subject_ref, "the patient")

    def first_display(code_obj):
        return code_obj.get("coding", [{}])[0].get("display", "") or code_obj.get("text", "")

    if resource_type == "observation":
        name = first_display(resource.get("code", {}))
        value = resource.get("valueQuantity", {}).get("value", "")
        unit = resource.get("valueQuantity", {}).get("unit", "")
        date = resource.get("effectiveDateTime", "")
        return f"Observation: {patient_name} had a {name} of {value} {unit} on {date}."

    elif resource_type == "condition":
        name = first_display(resource.get("code", {}))
        status = resource.get("clinicalStatus", {}).get("coding", [{}])[0].get("code", "")
        date = resource.get("onsetDateTime", "")
        return f"Condition: {patient_name} was diagnosed with {name} (status: {status}) on {date}."

    elif resource_type == "diagnosticreport":
        name = first_display(resource.get("code", {}))
        date = resource.get("effectiveDateTime", "")
        return f"Diagnostic Report: {name} was recorded for {patient_name} on {date}."

    elif resource_type == "patient":
        return f"Patient: {patient_name}"

    return ""


def process_bundle_file_to_sentences(input_file: str, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)
    with open(input_file, "r", encoding="utf-8") as f:
        bundle = json.load(f)

    sentences = []
    patient_name_map = {}

    if bundle.get("resourceType") == "Bundle" and "entry" in bundle:
        # First pass: extract patient names
        for entry in bundle["entry"]:
            resource = entry.get("resource", {})
            if resource.get("resourceType") == "Patient":
                patient_id = f"urn:uuid:{resource.get('id')}"
                name = extract_patient_name(resource)
                patient_name_map[patient_id] = name

        # Second pass: convert to plain English
        for entry in bundle["entry"]:
            resource = entry.get("resource", {})
            text = fhir_resource_to_text(resource, patient_name_map)
            if text:
                sentences.append(text)

    # Write result
    base_name = os.path.splitext(os.path.basename(input_file))[0]
    output_file = os.path.join(output_dir, f"{base_name}_pruned.txt")
    with open(output_file, "w", encoding="utf-8") as out:
        out.write("\n".join(sentences))

    print(f"Saved: {output_file}")


def process_all_files_to_sentences(input_dir: str, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)
    for filename in os.listdir(input_dir):
        if filename.endswith(".json"):
            input_path = os.path.join(input_dir, filename)
            process_bundle_file_to_sentences(input_path, output_dir)




process_all_files_to_sentences(in_file_glob, flat_file_path)


Saved: ./working/flat/Ammie189_Erdman779_4a615375-3591-59a9-5f2c-360da9a85972_pruned.txt
Saved: ./working/flat/Carlo647_Gislason620_b7ec4930-0a84-73bc-ffb7-8af376b4d991_pruned.txt
Saved: ./working/flat/Arnetta705_Lang846_28f969b1-98bb-c124-8dd0-d29beb5fa24b_pruned.txt
Saved: ./working/flat/Brent147_Considine820_19044bac-e685-a09d-2c79-769263eaac4e_pruned.txt
Saved: ./working/flat/Colin861_Medhurst46_57f13fc4-0882-fda3-e99a-7c8b8d0d086b_pruned.txt
Saved: ./working/flat/Alex454_White193_2f12e0bd-98b8-8ba2-643a-6aca9799cbb2_pruned.txt


## Setup the Gen AI with RAG

This section will use LlamaIndex to construct the vector store and tie to the LLM. 

In [3]:
!pip install llama-index
!pip install transformers
!pip install llama-index-embeddings-huggingface
!pip install llama-index-llms-ollama

I tried a couple of different models for doing the embedding, i.e. turning the flattened FHIR text into vectors. I would like to experement with others, but haven't had time. In the end, `BAAI/bge-large-en-v1.5` was too big for me to run on my local, so I did most of my testing with `BAAI/bge-small-en-v1.5`.

In [4]:

from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# loads BAAI/bge-small-en
# embed_model = HuggingFaceEmbedding()

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-large-en-v1.5")

# embed_model = HuggingFaceEmbedding(model_name="medicalai/ClinicalBERT")

/home/chapsk/llm/rag-fhir/RAG_on_FHIR/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from llama_index.llms.ollama import Ollama

# LLama 3 is running locally, using Ollama.
llm = Ollama(model="llama3.2:3b-instruct-q4_0", request_timeout=300)

In [35]:

from llama_index.core import (
    Settings,
    VectorStoreIndex,
    SimpleDirectoryReader,
    SummaryIndex # Assuming SummaryIndex is also in core, adjust if needed
)
# Assuming 'llm' and 'embed_model' are already defined LlamaIndex LLM and Embedding objects
# Configure global settings instead of using ServiceContext
Settings.llm = llm
Settings.embed_model = embed_model

# Now you can create indices or query engines, and they will use the models from Settings
# Example:
# documents = SimpleDirectoryReader(...).load_data()
# index = VectorStoreIndex.from_documents(documents) # Will use Settings.embed_model
# query_engine = index.as_query_engine() # Will use Settings.llm and Settings.embed_model


In [36]:
# This code loads the flat FHIR text files. 

documents = SimpleDirectoryReader(flat_file_path).load_data()
print(len(documents))

6


In [37]:
# Load those flat FHIR text files into the vector store.

vector_index = VectorStoreIndex.from_documents(documents, show_progress=True)


# if not os.path.exists(vector_store_file_path):
#     os.mkdir(vector_store_file_path)
# vector_index.vector_store.persist(f'{vector_store_file_path}/FHIR_RAG.vs')

Parsing nodes:   0%|          | 0/6 [00:00<?, ?it/s]

Generating embeddings: 100%|██████████| 1530/1530 [06:58<00:00,  3.65it/s]


## Actually do RAG

This is the code block that actually asks the questions of the LLM. 

In [39]:

from llama_index.core.query_engine import RetrieverQueryEngine



# STEP 3: Create retriever and query engine
retriever = vector_index.as_retriever()
query_engine = RetrieverQueryEngine.from_args(retriever=retriever, llm=llm)

# STEP 4: Ask a question
question = "What can you tell me about Ammie189 history?"
response = query_engine.query(question)

# STEP 5: Print the response
print("Answer:", response.response)

INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
Answer: Ammie189 appears to have a complex medical history. She has been diagnosed with various conditions at different times, including Educated to high school level (a finding), Essential hypertension, Primary fibromyalgia syndrome, Social isolation (which was resolved), History of tubal ligation, Stress (another finding that was resolved), Part-time employment (a finding that was resolved), Medication review due (a situation that was resolved), and Full-time employment (a finding that remains active). Additionally, Ammie189 has experienced various physical symptoms, such as pain, weight changes, and blood test results.
